# Finales Hauptexperiment: alle zehn Auswertungen als Texttabellen

## Eingaben und Ausgaben

Nach der einmaligen Vorbereitung mit `python 05_analyze.py --prepare` genügt **Run All**.
Die folgende Zelle zeigt Daten, Vorgängertabellen und neue Ausgaben. `DATA_DIR` und
`OUTPUT_DIR` dürfen auf eigene absolute Verzeichnisse zeigen. `TABLES` liest standardmäßig
die vorbereiteten historischen Tabellen; für selbst neu erzeugte Vorgängertabellen
kann hier `OUTPUT_DIR / "tables"` stehen. Eingaben und neue Ausgaben bleiben getrennt.


In [1]:
from pathlib import Path
from faktenretention.paths import PROJECT_ROOT
from faktenretention.text_rules import count_words
from faktenretention.notebook_context import setup_notebook
from faktenretention.io import relative_path, sha

# Hier können eigene absolute Daten- und Ausgabeverzeichnisse stehen.
DATA_DIR = PROJECT_ROOT / "data/prepared"
OUTPUT_DIR = PROJECT_ROOT / "results/notebooks"
# Historische Vorgängertabellen; für eigene Neuberechnungen OUTPUT_DIR / "tables" einsetzen.
TABLES = DATA_DIR / "tables"
setup_notebook(DATA_DIR, OUTPUT_DIR)
OUT = OUTPUT_DIR / "tables/08_tabellenexport"
FIG = OUTPUT_DIR / "figures/08_tabellenexport"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
print("Eingaben:", DATA_DIR / "qa")
print("Vorgängertabellen:", TABLES)
print("Neue Tabellen:", OUT)
print("Neue Grafiken:", FIG)


Eingaben: /home/edgar/Faktenretention_Notebook_Refactor/data/prepared/qa
Vorgängertabellen: /home/edgar/Faktenretention_Notebook_Refactor/data/prepared/tables
Neue Tabellen: /home/edgar/Faktenretention_Notebook_Refactor/results/notebooks/tables/08_tabellenexport
Neue Grafiken: /home/edgar/Faktenretention_Notebook_Refactor/results/notebooks/figures/08_tabellenexport


**Datenstand `aktuell_20260930` · 100 Artikel · sechs Verfahren · 500 → 300 → 150 → 75 Wörter.**

Dieses Notebook berechnet sämtliche angeforderten Werte aus den eingefrorenen
Einzeldaten. Die Tabellen erscheinen vollständig als kopierbarer Monospace-Text.
Unter **jeder** Tabelle stehen Nenner, Aggregation und Fehlwertbehandlung.
Zusätzlich entstehen eine gemeinsame Textdatei und CSV-Dateien mit ungerundeten Werten
im Ordner `OUTPUT_DIR/tables/08_tabellenexport`.

## Überblick

Die ausgeführte Rechnung ergibt **196 technische QA-Fehlwerte von 48.000 Positionen**
(0,4083 %). Der Nennerwechsel verändert **keine** Methodenrangposition; die
Längenadjustierung verändert **11 von 24 Positionen auf drei Stufen**. Der ursprüngliche
Scorer stimmt in **32/52**, der finale in **41/52** eindeutigen Kalibrierungsfällen
mit der Human-Referenz überein. Diese Fälle sind kein unabhängiger Test.

Am Ende steht zusätzlich eine automatisch berechnete Übersicht. Es werden keine
Pilotdaten und keine alten Plain-/Few-Shot-Ergebnisse gepoolt.

## Kontext und Methoden

- Artikelmittel gewichten jeden Artikel gleich. Zunächst wird pro Artikel gerechnet,
  erst danach gemittelt. Faktenquoten nach Entitätstyp gewichten dagegen die Fakten;
  diese andere Aggregation ist bei den Tabellen ausdrücklich angegeben.
- `K` = Retained; `V` = gültige QA-Bewertungen. `R_obs = K/V`,
  `R_20 = K/20`, `R_adj = R_20/(1+D)`. Der **Längenabschlag** ist
  `100 × mean(R_20 − R_adj)` in Prozentpunkten: derselbe QA-Nenner 20.
- `D = (max(U−W,0) + max(W−O,0))/T`. Wortzähler: Python-RegEx `\b\w+\b`.
  Inklusive Fenster: 475–525, 285–315, 142–158 und 71–79 Wörter.
- Sämtliche Intervalle: **10.000 Artikelbootstrap-Ziehungen**, `np.random.default_rng(32)`,
  95-%-Perzentilintervall (2,5 % und 97,5 %, lineare Quantile).
  Eine gemeinsame Ziehungsmatrix erhält Artikelpaarung, Fakten, Verfahren und Stufen.
  In Korrelationen werden die Ränge nach jeder Ziehung neu berechnet.
- Mittelwerte, Differenzen, Korrelationen und Ränge verwenden ungerundete Daten;
  die drei Retentionsrangfolgen verwenden sogar exakte rationale Mittelwerte.
  Exakte Gleichstände erhalten Wettbewerbsränge (1, 1, 3). Die Anzeige wird erst zuletzt gerundet.
- Retention und Anteile werden als **Prozent**, Differenzen als **Prozentpunkte**
  angezeigt. Die sechs Evaluationsmetriken bleiben auf ihrer ursprünglichen Score-Skala.
  `fehlend` kennzeichnet nicht auswertbare Ergebnisse; es bedeutet nicht null.

### Annahmen und bekannte Besonderheiten

`Unsupported` ist im Hauptexperiment kein eigenes Quelllabel. Die tatsächliche Kategorie
heißt `reference_discordant`; sie wird hier als Unsupported/reference_discordant
ausgewiesen und darf nicht als nachgewiesene Halluzination interpretiert werden.
Nicht Retained bedeutet Lost oder reference_discordant; technisch fehlend bleibt separat.

Zwei leere Zero-Shot-CoT-Ausgaben bleiben mit den dokumentierten Nullwerten der
End-to-end-Evaluation enthalten. Ihre regelbasierten NOT_FOUND-Labels sind Lost.
Bei Artikel 91 / Zero-Shot-CoT / 75 Wörter ist der tatsächliche Stufeneingang leer:
der lokale SummaC-Wert wird deshalb **fehlend** geführt, auch wenn im Rohexport ein Score steht.

Die Kalibrierung vergleicht beide Bewertungsregeln auf denselben historischen Antworten.
Primär werden eindeutige menschliche Konsenslabels verwendet; Ties und `unclear` sind
ausgeschlossen. Das ist Leistung auf **Kalibrierungsfällen**, kein unabhängiger Test
der gesamten finalen Pipeline. Intervalle sind punktweise und nicht mehrfachvergleichskorrigiert.

### Erneut ausführen

Notebook im Ordner mit den mitgelieferten Daten starten und alle Zellen ausführen.
Benötigt: Python, numpy, pandas, scipy, nbformat, nbclient, nbconvert und ipykernel.
Keine Modellaufrufe, API-Schlüssel oder Netzwerkzugriffe erforderlich.

## Bereits geprüfte Ergebnisse verwenden

QA-Labels, Retentionsmittel und Intervalle, Entitätstyp-Quotientenschätzer und Sekundärmetrikmittel
werden aus den vorher ausgeführten Detailnotebooks übernommen. Die übrigen Zusatzanalysen
(unter anderem gemeinsame Bootstrap-Kontraste, exakte Ranggleichstände und gepoolte Übergänge)
behalten ihre eigenen dokumentierten Schätzer. Die bisherigen unabhängigen Kontrollabgleiche bleiben erhalten.

## Daten und Reproduzierbarkeit

Quellen prüfen, Rohantworten neu klassifizieren und Artikelwerte bilden.

In [2]:
from pathlib import Path
from fractions import Fraction
from collections import Counter
import gzip, hashlib, importlib.util, importlib.metadata, json, platform, re
from textwrap import fill
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, rankdata

BASE = DATA_DIR
VERSION, SEED, B = "aktuell_20260930", 32, 10_000
from faktenretention.protocol import METHODS
from faktenretention.protocol import LABELS as NAMES
from faktenretention.protocol import STAGES
from faktenretention.protocol import WINDOWS
CELL = ["article_id", "method", "stage_target_words"]
FACTKEY = ["article_id", "canonical_fact_id"]
KEY = CELL + ["canonical_fact_id"]
R, L, D_LABEL, M = "retained", "lost", "reference_discordant", "technical_missing"
NON = {L, D_LABEL}
QA_DATA = DATA_DIR / "qa"
MET_DATA = DATA_DIR / "metrics"
VAL_DATA = DATA_DIR / "human_qa"
OUT.mkdir(parents=True, exist_ok=True)
TEXT_FILE = OUT/"alle_zehn_auswertungen.txt"
TEXT_FILE.write_text("Finales Hauptexperiment – zehn Auswertungen\n"
                     "10.000 Artikelbootstrap-Ziehungen, Seed 32, 95-%-Perzentilintervalle.\n\n", encoding="utf-8")
table_registry, sources, validation_checks = [], [], []

from faktenretention.io import read_table as read

def record_source(path, role, expected=None):
    path = Path(path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if expected is not None:
        assert digest == expected, f"Quellenhash verändert: {path}"
    rel = relative_path(path, BASE)
    if rel not in {x["Datei"] for x in sources}:
        sources.append({"Datei":rel, "SHA256":digest, "Rolle":role})

def frozen_input(root, relative):
    manifest_path = root/"source_manifest.json"
    record_source(manifest_path, "Quellenmanifest")
    manifest = json.loads(manifest_path.read_text())
    entry = next(e for e in manifest["files"] if e["file"] == relative)
    path = root/relative
    record_source(path, "eingefrorene Eingabe", entry["sha256"])
    if path.suffix == ".gz" and "original_sha256" in entry:
        assert hashlib.sha256(gzip.decompress(path.read_bytes())).hexdigest() == entry["original_sha256"]
    return path

def numeric(series):
    return pd.to_numeric(series.replace("", np.nan), errors="raise")

def bool_values(series):
    assert series.astype(str).str.lower().isin(["true", "false"]).all()
    return series.astype(str).str.lower().eq("true")

def table(frame, name, title, note, digits=4):
    # Print all rows as copyable text; retain full precision in the CSV export.
    assert name not in table_registry
    table_registry.append(name)
    frame.to_csv(OUT/f"{name}.csv", index=False, na_rep="fehlend")
    shown = frame.copy()
    for col in shown.select_dtypes(include="floating"):
        shown[col] = shown[col].map(lambda v: "fehlend" if pd.isna(v) else f"{v:.{digits}f}")
    shown = shown.fillna("fehlend")
    text = title + "\n" + shown.to_string(index=False, max_rows=None, max_cols=None) + "\n\n" + fill(note,width=105) + "\n"
    print(text)
    with TEXT_FILE.open("a", encoding="utf-8") as stream:
        stream.write(text + "\n")

def percentile_interval(values):
    finite = np.asarray(values)[np.isfinite(values)]
    return np.quantile(finite, [.025,.975], method="linear") if len(finite) else np.array([np.nan,np.nan])


In [3]:
facts = read(frozen_input(QA_DATA, "shared/canonical_facts.csv.gz"))
summaries = read(frozen_input(QA_DATA, VERSION+"/summaries.csv.gz"))
raw_qa = read(frozen_input(QA_DATA, VERSION+"/answers.csv.gz")).rename(columns={"summary_prompt_variant":"method"})
states_path = frozen_input(QA_DATA, VERSION+"/generation_states.jsonl.gz")
scorer_path = frozen_input(QA_DATA, "code/qa_scorer_snapshot.py")
freeze_path = frozen_input(QA_DATA, "shared/FINAL_100_EXPERIMENT_FREEZE.md")
merge_path = frozen_input(QA_DATA, VERSION+"/provenienz/result_merge_manifest.json")
merge_info = json.loads(merge_path.read_text())
assert merge_info["retained_historical_rows"] == 400
assert merge_info["replaced_plain_rows"] == merge_info["replaced_few_shot_rows"] == 100
assert merge_info["historical_plain_or_few_shot_rows_in_new_table"] == 0
IDS = sorted(facts.article_id.unique())
N = len(IDS)
assert N == 100 and len(facts) == 2000 and facts.canonical_fact_id.is_unique
assert facts.groupby("article_id").size().eq(20).all() and facts.entity_type.ne("").all()
assert len(summaries) == 600 and not summaries.duplicated(["article_id","method"]).any()
assert set(summaries[["article_id","method"]].itertuples(index=False,name=None)) == {(a,m) for a in IDS for m in METHODS}
phase = {m: "plain_fewshot_rerun_20260930" if m in ["plain","few_shot_cot"]
         else "final_100_frozen_temperature_0" for m in METHODS}
assert summaries.collection_phase.eq(summaries.method.map(phase)).all()
expected_keys = {(a,m,t,f) for a,f in facts[FACTKEY].itertuples(index=False,name=None) for m in METHODS for t in STAGES}
assert len(raw_qa) == 48000 and not raw_qa.duplicated(KEY).any()
assert set(raw_qa[KEY].itertuples(index=False,name=None)) == expected_keys

rows = []
for t in STAGES:
    previous = "article" if t == 500 else f"summary_{STAGES[STAGES.index(t)-1]}"
    sub = summaries[["article_id","method",f"summary_{t}", previous,
                     f"summary_{t}_word_count",f"summary_{t}_complete_sentence_success",
                     f"summary_{t}_final_window_success"]].copy()
    sub.columns = ["article_id","method","summary","stage_input","W_stored","sentence_stored","window_stored"]
    sub["stage_target_words"] = t
    rows.append(sub)
outputs = pd.concat(rows, ignore_index=True)
outputs["W"] = outputs.summary.map(count_words)
outputs["U"] = outputs.stage_target_words.map(lambda t: WINDOWS[t][0])
outputs["O"] = outputs.stage_target_words.map(lambda t: WINDOWS[t][1])
outputs["violation_words"] = (outputs.U-outputs.W).clip(lower=0)+(outputs.W-outputs.O).clip(lower=0)
outputs["D"] = outputs.violation_words/outputs.stage_target_words
outputs["window_status"] = np.select([outputs.W.lt(outputs.U), outputs.W.gt(outputs.O)], ["unterhalb","oberhalb"], default="innerhalb")
outputs["complete_sentence"] = bool_values(outputs.sentence_stored)
outputs["empty_output"] = outputs.summary.str.strip().eq("")
outputs["empty_input"] = outputs.stage_input.str.strip().eq("")
assert outputs.W.eq(outputs.W_stored.astype(int)).all()
assert bool_values(outputs.window_stored).eq(outputs.window_status.eq("innerhalb")).all()
output_lookup = outputs.set_index(CELL)
with gzip.open(states_path, "rt") as stream:
    states = [json.loads(line) for line in stream]
assert len(states) == 600 and len({(s["article_id"],s["method"]) for s in states}) == 600
for rec in states:
    for stage in rec["state"]["stages"]:
        row = output_lookup.loc[(rec["article_id"],rec["method"],stage["target_words"])]
        selected = stage["selected"]
        assert row.summary == selected["response"] and row.stage_input == stage["input_text"]
        assert row.W == selected["response_word_count"]
        assert row.complete_sentence == selected["ends_with_complete_sentence"]
        if "word_window_violation_rate" in selected:
            assert np.isclose(row.D, selected["word_window_violation_rate"], rtol=0, atol=1e-12)

from faktenretention import scoring as scorer
qa = raw_qa.merge(facts[FACTKEY+["entity_type","reference_answer"]], on=FACTKEY,
                  validate="many_to_one", suffixes=("","_canonical"))
assert qa.entity_type.eq(qa.entity_type_canonical).all()
assert qa.reference_answer.eq(qa.reference_answer_canonical).all()
qa = qa.drop(columns=["entity_type_canonical","reference_answer_canonical"])
qa = qa.merge(outputs[CELL+["summary"]], on=CELL, validate="many_to_one")
qa["technical_missing"] = qa.error.ne("")
qa["valid"] = ~qa.technical_missing
assert qa.loc[qa.valid,"answer"].str.strip().ne("").all()
assert all(scorer.support_is_grounded(s,a,e) for s,a,e in qa.loc[qa.valid,["summary","answer","support"]].itertuples(index=False,name=None))
labels_path = TABLES/f"03_qa_retention/qa_labels_reproduziert.csv.gz"
record_source(labels_path, "Abgleich vorhandener Auswertung")
validated_labels = read(labels_path)[KEY+["label"]]
qa = qa.merge(validated_labels, on=KEY, validate="one_to_one")
qa["is_retained"] = qa.label.eq(R)
assert qa.label.eq(M).equals(qa.technical_missing)
assert set(qa.label.unique()) == {R,L,D_LABEL,M}
qa["is_lost"] = qa.label.eq(L)


In [4]:
qa["is_discordant"] = qa.label.eq(D_LABEL)
counts = qa.groupby(CELL).agg(K=("is_retained","sum"), V=("valid","sum"),
                             n_missing=("technical_missing","sum"), n_lost=("is_lost","sum"),
                             n_discordant=("is_discordant","sum"), n_expected=("label","size")).reset_index()
assert counts.n_expected.eq(20).all()
assert counts.K.add(counts.n_lost).add(counts.n_discordant).eq(counts.V).all()
articles = outputs.merge(counts,on=CELL,validate="one_to_one")
articles["R_obs"] = articles.K.div(articles.V.where(articles.V.gt(0)))
articles["R_20"] = articles.K/20
articles["R_adj"] = articles.R_20/(1+articles.D)
assert (articles.R_adj <= articles.R_20).all()
articles.drop(columns=["summary","stage_input"]).to_csv(OUT/"artikelwerte_laenge_qa.csv",index=False)
qa[KEY+["entity_type","label"]].to_csv(OUT/"qa_labels.csv.gz",index=False)

rng = np.random.default_rng(SEED)
draws = rng.integers(0,N,size=(B,N))
weights = np.zeros((B,N),dtype=np.int16)
np.add.at(weights,(np.repeat(np.arange(B),N),draws.ravel()),1)
assert np.all(weights.sum(axis=1) == N)
pass  # Große Bootstrap-Zwischendatei wird nicht benötigt.

def vector(method, stage, metric):
    return articles.loc[articles.method.eq(method)&articles.stage_target_words.eq(stage)].set_index("article_id").reindex(IDS)[metric].to_numpy(dtype=float)

def estimate_mean(values):
    values = np.asarray(values,dtype=float)
    valid = np.isfinite(values)
    numerator = weights @ np.where(valid,values,0.)
    denominator = weights @ valid.astype(float)
    boot = np.divide(numerator,denominator,out=np.full(B,np.nan),where=denominator>0)
    lo,hi = percentile_interval(boot)
    return {"mean":float(values[valid].mean()) if valid.any() else np.nan,
            "low":lo,"high":hi,"n":int(valid.sum()),"n_boot_valid":int(np.isfinite(boot).sum())},boot

def estimate_ratio(frame, numerator, denominator):
    # Pooled count ratio; all facts/methods of each article resampled together.
    per_article = frame.groupby("article_id")[[numerator,denominator]].sum().reindex(IDS,fill_value=0)
    totals = per_article.sum()
    boot_counts = weights @ per_article.to_numpy(dtype=float)
    boot = np.divide(boot_counts[:,0],boot_counts[:,1],out=np.full(B,np.nan),where=boot_counts[:,1]>0)
    lo,hi = percentile_interval(boot)
    return {"count":int(totals.iloc[0]),"den":int(totals.iloc[1]),
            "mean":totals.iloc[0]/totals.iloc[1] if totals.iloc[1] else np.nan,
            "low":lo,"high":hi,"n_articles":int(per_article.iloc[:,1].gt(0).sum()),
            "n_boot_valid":int(np.isfinite(boot).sum())}

validated_path = TABLES/f"04_laenge_und_adjustierte_retention/retention_und_drei_rangfolgen_24_gruppen.csv"
record_source(validated_path, "Abgleich vorhandener Auswertung")
validated_retention = read(validated_path).set_index(["method","stage_target_words"])
retention_stats, exact_means = {}, {}
for m in METHODS:
    for t in STAGES:
        sub = articles.loc[articles.method.eq(m)&articles.stage_target_words.eq(t)]
        for metric in ["R_obs","R_20","R_adj"]:
            previous = validated_retention.loc[(m,t)]
            prefix = {"R_obs":"R_obs","R_20":"R_zero","R_adj":"R_adj"}[metric]
            retention_stats[m,t,metric] = {"mean":previous[prefix+"_mean"],
                "low":previous[prefix+"_ci_low"],"high":previous[prefix+"_ci_high"],
                "n":int(previous.n_articles),"n_boot_valid":B}
            exact = []
            for row in sub.itertuples():
                if metric == "R_obs" and row.V > 0: exact.append(Fraction(int(row.K),int(row.V)))
                elif metric == "R_20": exact.append(Fraction(int(row.K),20))
                elif metric == "R_adj": exact.append(Fraction(int(row.K)*int(t),20*(int(t)+int(row.violation_words))))
            exact_means[m,t,metric] = sum(exact,Fraction(0))/len(exact) if exact else None

def rank(method, stage, metric):
    own = exact_means[method,stage,metric]
    return 1+sum(v is not None and v > own for (m,t,k),v in exact_means.items() if t==stage and k==metric) if own is not None else np.nan

print(f"Geprüft: {N} Artikel, {len(articles)} Ausgaben, {len(qa)} QA-Positionen; "
      f"{int(qa.technical_missing.sum())} technische Fehlwerte.")


Geprüft: 100 Artikel, 2400 Ausgaben, 48000 QA-Positionen; 196 technische Fehlwerte.


## Ergebnisse

### 1. Fehlende QA-Bewertungen und Nennervergleich

In [5]:
rows = []
for t in STAGES:
    for m in METHODS:
        sub = articles[articles.method.eq(m)&articles.stage_target_words.eq(t)]
        row = {"Verfahren":NAMES[m],"Wörter":t,"QA_gültig":int(sub.V.sum()),"QA_fehlend":int(sub.n_missing.sum())}
        for metric,title in [("R_obs","R_gültig"),("R_20","R_20")]:
            s = retention_stats[m,t,metric]
            row.update({f"{title}_%":100*s["mean"],f"{title}_KI_lo":100*s["low"],f"{title}_KI_hi":100*s["high"],f"Rang_{title}":rank(m,t,metric)})
        row["n_Artikel_R_gültig"] = retention_stats[m,t,"R_obs"]["n"]
        rows.append(row)
t1 = pd.DataFrame(rows)
table(t1,"01_qa_nenner_retention","1. QA-Missingness, Retention (%) und Ränge",
      "Nenner: je Artikel V gültige QA-Urteile bzw. fest 20; je Gruppe 2.000 erwartete QA-Positionen. "
      "Aggregation: ungewichtetes Mittel der Artikelquoten, Ränge je Stufe. "
      "Fehlwerte: technische QA-Fehler aus V ausgeschlossen; bei Nenner 20 liefern sie keinen Retained-Beitrag. "
      "V=0 ergibt fehlende R_gültig; gültige Artikelanzahl ist ausgewiesen. KI: gemeinsamer Artikelbootstrap.")


1. QA-Missingness, Retention (%) und Ränge
    Verfahren  Wörter  QA_gültig  QA_fehlend R_gültig_% R_gültig_KI_lo R_gültig_KI_hi  Rang_R_gültig  R_20_% R_20_KI_lo R_20_KI_hi  Rang_R_20  n_Artikel_R_gültig
       Lead-k     500       1995           5    75.5868        72.8315        78.2553              1 75.4000    72.6500    78.0500          1                 100
        Plain     500       1990          10    54.8915        51.5339        58.1148              6 54.6000    51.2500    57.8012          6                 100
Zero-Shot-CoT     500       1993           7    71.8883        69.1412        74.5989              3 71.6500    68.9000    74.3500          3                 100
 Few-Shot-CoT     500       1984          16    64.1906        61.2117        67.0335              5 63.7000    60.7000    66.5500          5                 100
     Auto-CoT     500       1988          12    71.3371        68.6136        73.9270              4 70.9500    68.2000    73.5500          4      

### 2. Längeneinhaltung

In [6]:
length_rows, status_rows = [], []
for t in STAGES:
    for m in METHODS:
        sub = articles[articles.method.eq(m)&articles.stage_target_words.eq(t)]
        q1,med,q3 = sub.W.quantile([.25,.5,.75],interpolation="linear")
        length_rows.append({"Verfahren":NAMES[m],"Wörter":t,"n":len(sub),"MW":sub.W.mean(),
                            "SD":sub.W.std(ddof=1),"Q1":q1,"Median":med,"Q3":q3})
        row = {"Verfahren":NAMES[m],"Wörter":t,"Fenster":f"{WINDOWS[t][0]}–{WINDOWS[t][1]}","n":len(sub)}
        for label in ["unterhalb","innerhalb","oberhalb"]:
            count = int(sub.window_status.eq(label).sum())
            row[f"n_{label}"] = count
            row[f"{label}_%"] = 100*count/len(sub)
        row.update({"D_MW":sub.D.mean(),"n_Satzende":int(sub.complete_sentence.sum()),
                    "Satzende_%":100*sub.complete_sentence.mean(),"n_Satzende_erfasst":int(sub.complete_sentence.notna().sum())})
        status_rows.append(row)
table(pd.DataFrame(length_rows),"02a_wortzahlen","2a. Tatsächliche Wortzahl",
      "Nenner: 100 Ausgaben je Verfahren/Stufe. Aggregation: Artikelverteilung; SD mit ddof=1; "
      "Quartile mit linearer Interpolation. Fehlwerte: keine Wortzahl fehlt; Leertexte zählen als 0 Wörter.")
table(pd.DataFrame(status_rows),"02b_wortfenster_satzende","2b. Wortfenster, normalisierte Abweichung und vollständiges Satzende",
      "Nenner: 100 Ausgaben je Gruppe, einschließlich Leertexte; Satzende-Anteil unter den erfassten "
      "Satzende-Flags (hier jeweils 100). Aggregation: Ausgabenzählung und Artikelmittel von D; "
      "Fenstergrenzen inklusive. Fehlwerte: keine Flags fehlen. Satzende aus gespeicherter Generierungsdiagnose, "
      "keine nachträgliche linguistische Neubewertung.")


2a. Tatsächliche Wortzahl
    Verfahren  Wörter   n       MW      SD       Q1   Median       Q3
       Lead-k     500 100 500.0000  0.0000 500.0000 500.0000 500.0000
        Plain     500 100 209.3100 47.3047 177.0000 209.5000 233.5000
Zero-Shot-CoT     500 100 345.2900 50.0955 306.0000 344.5000 380.0000
 Few-Shot-CoT     500 100 305.5300 42.2143 279.5000 306.0000 334.2500
     Auto-CoT     500 100 349.7000 54.4420 310.7500 348.5000 383.2500
 Automate-CoT     500 100 392.7000 43.7448 359.0000 390.5000 423.7500
       Lead-k     300 100 300.0000  0.0000 300.0000 300.0000 300.0000
        Plain     300 100 175.7500 41.8391 146.0000 169.0000 198.0000
Zero-Shot-CoT     300 100 285.6700 40.6279 258.5000 287.0000 312.0000
 Few-Shot-CoT     300 100 270.0700 35.5523 246.7500 269.5000 292.0000
     Auto-CoT     300 100 302.9800 49.3622 272.0000 302.5000 332.5000
 Automate-CoT     300 100 312.9500 46.3213 278.0000 310.0000 346.2500
       Lead-k     150 100 150.0000  0.0000 150.0000 150.0000 150

### 3. Längenadjustierte Retention und Rangwechsel

In [7]:
rows, changes = [], []
for t in STAGES:
    for m in METHODS:
        s = retention_stats[m,t,"R_adj"]
        penalty = vector(m,t,"R_20")-vector(m,t,"R_adj")
        rows.append({"Verfahren":NAMES[m],"Wörter":t,"R_adj_%":100*s["mean"],"KI_lo_%":100*s["low"],"KI_hi_%":100*s["high"],
                     "Abschlag_ggü_R20_pp":100*np.mean(penalty),"Rang_gültig":rank(m,t,"R_obs"),
                     "Rang_20":rank(m,t,"R_20"),"Rang_adj":rank(m,t,"R_adj"),"n":s["n"]})
    changes.append({"Wörter":t,
                    "Positionen_Nennerwechsel":sum(rank(m,t,"R_obs")!=rank(m,t,"R_20") for m in METHODS),
                    "Positionen_Längenadjustierung":sum(rank(m,t,"R_20")!=rank(m,t,"R_adj") for m in METHODS)})
t3 = pd.DataFrame(rows)
changes = pd.DataFrame(changes)
table(t3,"03a_adjustierung_rangfolgen","3a. Adjustierte Retention (%), Abschlag (pp) und drei Ränge",
      "Nenner: R_adj und Abschlag verwenden je Artikel 20 QA-Positionen; R_gültig verwendet V. "
      "Aggregation: Adjustierung pro Artikel, danach ungewichtetes Artikelmittel; "
      "Abschlag = 100 × mean(R_20 − R_adj), nicht R_gültig − R_adj. "
      "Fehlwerte: technische QA-Fehler geben keinen Retained-Beitrag im Nenner 20. Alle 100 Artikel bleiben enthalten.")
table(changes,"03b_rangwechsel_stufen","3b. Anzahl veränderter Methodenrangpositionen je Stufe",
      "Nenner: sechs Methodenpositionen je Stufe. Aggregation: Anzahl Methoden mit geändertem Rang, "
      "nicht Summe der zurückgelegten Rangplätze. Nennerwechsel: R_gültig → R_20; "
      "Längenadjustierung: R_20 → R_adj. Fehlwerte: keine Gruppenränge fehlen.")
change_total = pd.DataFrame([
    {"Änderung":"Nennerwechsel R_gültig→R_20", "Positionen":int(changes.Positionen_Nennerwechsel.sum()),
     "von_Positionen":24,"betroffene_Stufen":int(changes.Positionen_Nennerwechsel.gt(0).sum()),"von_Stufen":4},
    {"Änderung":"Längenadjustierung R_20→R_adj", "Positionen":int(changes.Positionen_Längenadjustierung.sum()),
     "von_Positionen":24,"betroffene_Stufen":int(changes.Positionen_Längenadjustierung.gt(0).sum()),"von_Stufen":4}])
table(change_total,"03c_rangwechsel_gesamt","3c. Rangwechsel insgesamt",
      "Nenner: 24 Methoden-Stufen-Positionen bzw. vier Stufen. Aggregation: Summe geänderter "
      "Positionen; eine Stufe ist betroffen, sobald mindestens eine Position wechselt. Fehlwerte: keine.")


3a. Adjustierte Retention (%), Abschlag (pp) und drei Ränge
    Verfahren  Wörter R_adj_% KI_lo_% KI_hi_% Abschlag_ggü_R20_pp  Rang_gültig  Rang_20  Rang_adj   n
       Lead-k     500 75.4000 72.6500 78.0500              0.0000            1        1         1 100
        Plain     500 36.1535 33.7219 38.5280             18.4465            6        6         6 100
Zero-Shot-CoT     500 57.4682 54.9688 59.9583             14.1818            3        3         3 100
 Few-Shot-CoT     500 47.7908 45.4890 50.0418             15.9092            5        5         5 100
     Auto-CoT     500 57.3919 54.8497 59.9263             13.5581            4        4         4 100
 Automate-CoT     500 63.1115 60.4491 65.7778              9.8885            2        2         2 100
       Lead-k     300 49.3000 46.6000 52.0000              0.0000            6        6         5 100
        Plain     300 37.2912 34.4148 40.0709             12.4588            5        5         6 100
Zero-Shot-CoT     300 

### 4. Gepaarte Methodenunterschiede

In [8]:
from faktenretention.protocol import CONTRASTS
contrast_records = []
for metric,title in [("R_obs","gültiger Nenner"),("R_20","fester Nenner 20"),("R_adj","längenadjustiert")]:
    rows = []
    for t in STAGES:
        for a,b in CONTRASTS:
            x,y = vector(a,t,metric),vector(b,t,metric)
            diff = np.where(np.isfinite(x)&np.isfinite(y),x-y,np.nan)
            s,_ = estimate_mean(diff)
            rows.append({"Wörter":t,"Vergleich":f"{NAMES[a]} minus {NAMES[b]}","Δ_pp":100*s["mean"],
                         "KI_lo_pp":100*s["low"],"KI_hi_pp":100*s["high"],"n_Paare":s["n"]})
            contrast_records.append({"method_a":a,"method_b":b,"stage_target_words":t,"metric":metric,**s})
    table(pd.DataFrame(rows),f"04_kontraste_{metric}",f"4. Gepaarte Differenzen: {title}",
          "Nenner: vollständige gemeinsame Artikelpaare je Vergleich, n_Paare angegeben; QA-Nenner wie im Tabellentitel. "
          "Aggregation: Mittel artikelweiser Differenzen in Prozentpunkten. "
          "Fehlwerte: Paar nur bei zwei gültigen Artikelwerten; beide Verfahren mit derselben Artikelziehung resampelt. "
          "Technische QA-Fehler folgen der jeweiligen Retentionsdefinition.")
top_rows = []
for t in STAGES:
    order = sorted(METHODS,key=lambda m:exact_means[m,t,"R_adj"],reverse=True)
    a,b = order[:2]
    s,_ = estimate_mean(vector(a,t,"R_adj")-vector(b,t,"R_adj"))
    top_rows.append({"Wörter":t,"Erstplatziert":NAMES[a],"Zweitplatziert":NAMES[b],"Δ_pp":100*s["mean"],
                     "KI_lo_pp":100*s["low"],"KI_hi_pp":100*s["high"],"n_Paare":s["n"]})
top_pairs = pd.DataFrame(top_rows)
table(top_pairs,"04_spitzenvergleich_adjustiert","4. Erst- minus zweitplatziertes Verfahren nach R_adj",
      "Nenner: 100 gemeinsame Artikelpaare je Stufe; QA-Nenner 20. Aggregation: Mittel der gepaarten "
      "adjustierten Differenzen in pp. Fehlwerte: keine Artikelwerte fehlen. Methoden nach ungerundeten "
      "Gruppenpunktschätzern ausgewählt; Intervall punktweise, ohne Korrektur für diese datenabhängige Auswahl.")


4. Gepaarte Differenzen: gültiger Nenner
 Wörter                   Vergleich    Δ_pp KI_lo_pp KI_hi_pp  n_Paare
    500   Zero-Shot-CoT minus Plain 16.9968  14.2943  19.7678      100
    500    Few-Shot-CoT minus Plain  9.2991   6.7543  11.8834      100
    500 Auto-CoT minus Few-Shot-CoT  7.1465   4.8935   9.4813      100
    500 Automate-CoT minus Auto-CoT  2.0389   0.0779   4.0209      100
    300   Zero-Shot-CoT minus Plain 17.8430  15.1131  20.5747      100
    300    Few-Shot-CoT minus Plain 11.0322   8.5128  13.5718      100
    300 Auto-CoT minus Few-Shot-CoT  7.1193   4.9803   9.3177      100
    300 Automate-CoT minus Auto-CoT  0.9564  -1.1331   2.9819      100
    150   Zero-Shot-CoT minus Plain  8.5231   5.5866  11.3254      100
    150    Few-Shot-CoT minus Plain  5.7193   3.2754   8.1465      100
    150 Auto-CoT minus Few-Shot-CoT  7.1769   4.5444   9.8325      100
    150 Automate-CoT minus Auto-CoT -4.0509  -6.4974  -1.6739      100
     75   Zero-Shot-CoT minus Plain 

### 5. QA-Labels

In [9]:
rows = []
for t in STAGES:
    for m in METHODS:
        sub = qa[qa.method.eq(m)&qa.stage_target_words.eq(t)]
        row = {"Verfahren":NAMES[m],"Wörter":t,"Nenner":len(sub)}
        for label,title in [(R,"Retained"),(L,"Lost"),(D_LABEL,"U_ref"),(M,"Fehlend")]:
            n = int(sub.label.eq(label).sum())
            row[f"n_{title}"] = n
            row[f"{title}_%"] = 100*n/len(sub)
        rows.append(row)
qa_labels_table = pd.DataFrame(rows)
table(qa_labels_table,"05_qa_labels","5. QA-Labels: Anzahlen und Anteile (%)",
      "Nenner: alle 2.000 erwarteten QA-Positionen je Verfahren/Stufe, einschließlich technischer Fehlwerte. "
      "Aggregation: gepoolte Faktenzählung; diese Anteile sind keine Mittel artikelweiser Retention. "
      "U_ref = Unsupported/reference_discordant (tatsächliches Quelllabel reference_discordant). "
      "Fehlwerte: eigene Kategorie; die vier Kategorien partitionieren den Nenner vollständig.")


5. QA-Labels: Anzahlen und Anteile (%)
    Verfahren  Wörter  Nenner  n_Retained Retained_%  n_Lost  Lost_%  n_U_ref U_ref_%  n_Fehlend Fehlend_%
       Lead-k     500    2000        1508    75.4000     400 20.0000       87  4.3500          5    0.2500
        Plain     500    2000        1092    54.6000     670 33.5000      228 11.4000         10    0.5000
Zero-Shot-CoT     500    2000        1433    71.6500     346 17.3000      214 10.7000          7    0.3500
 Few-Shot-CoT     500    2000        1274    63.7000     464 23.2000      246 12.3000         16    0.8000
     Auto-CoT     500    2000        1419    70.9500     329 16.4500      240 12.0000         12    0.6000
 Automate-CoT     500    2000        1460    73.0000     312 15.6000      217 10.8500         11    0.5500
       Lead-k     300    2000         986    49.3000     911 45.5500      100  5.0000          3    0.1500
        Plain     300    2000         995    49.7500     745 37.2500      250 12.5000         10    0.500

### 6. Verlustmuster je Verfahren

In [10]:
PATHKEY = ["article_id","method","canonical_fact_id"]
paths = qa.pivot(index=PATHKEY,columns="stage_target_words",values="label").reindex(columns=STAGES).reset_index()
paths = paths.merge(facts[FACTKEY+["entity_type"]],on=FACTKEY,validate="many_to_one")
assert len(paths)==12000 and paths[STAGES].notna().all().all()

def classify_path(seq):
    first_index = next((i for i,label in enumerate(seq) if label in NON),None)
    if first_index is None:
        return ("offen" if M in seq else "durchgehend", "nicht_im_Risikoset")
    if M in seq[:first_index]:
        return "offen", "nicht_im_Risikoset"
    first = str(STAGES[first_index])
    if first_index == len(STAGES)-1:
        return first,"nicht_im_Risikoset"
    later = seq[first_index+1:]
    if R in later: return first,"später_Retained"
    if M in later: return first,"offen"
    return first,"durchgehend_nicht_Retained"

assert classify_path([R,M,L,R])==("offen","nicht_im_Risikoset")
assert classify_path([R,L,M,L])==("300","offen")
assert classify_path([R,R,R,L])==("75","nicht_im_Risikoset")
assert classify_path([L,M,R,L])==("500","später_Retained")
classification = [classify_path(seq) for seq in paths[STAGES].itertuples(index=False,name=None)]
paths[["first","later"]] = pd.DataFrame(classification,index=paths.index)
paths.to_csv(OUT/"faktenlaufbahnen.csv",index=False)
rows = []
for m in METHODS:
    sub = paths[paths.method.eq(m)]
    for cat,title in [(str(t),f"erstmals {t}") for t in STAGES]+[("durchgehend","durchgehend Retained"),("offen","erstes Nicht-Retained unklar")]:
        n = int(sub.first.eq(cat).sum())
        rows.append({"Verfahren":NAMES[m],"Verlauf":title,"Anzahl":n,"Nenner":len(sub),"Anteil_%":100*n/len(sub)})
first_table = pd.DataFrame(rows)
assert first_table.groupby("Verfahren").Anzahl.sum().eq(2000).all()
table(first_table,"06a_erste_nicht_retained_stufe","6a. Erste Nicht-Retained-Stufe und durchgehend Retained",
      "Nenner: alle 2.000 Ausgangsfakten/Laufbahnen pro Verfahren. Aggregation: Faktenzählung über "
      "500→300→150→75; Nicht-Retained = Lost oder reference_discordant. "
      "Fehlwerte: erste Stufe unklar, wenn ein Fehlwert vor dem ersten beobachteten Nicht-Retained liegt, "
      "oder bei nur Retained/Fehlwert ohne beobachtetes Nicht-Retained. Fehlwerte danach ändern die erste Zuordnung nicht.")
rows = []
for m in METHODS:
    sub = paths[paths.method.eq(m)&paths.first.isin(["500","300","150"])]
    for cat,title in [("durchgehend_nicht_Retained","danach durchgehend nicht Retained"),
                      ("später_Retained","später mindestens einmal Retained"),("offen","offen wegen späterem Fehlwert")]:
        n = int(sub.later.eq(cat).sum())
        rows.append({"Verfahren":NAMES[m],"Folgeverlauf":title,"Anzahl":n,"Nenner":len(sub),
                     "Anteil_%":100*n/len(sub) if len(sub) else np.nan})
persistence_table = pd.DataFrame(rows)
assert persistence_table.groupby("Verfahren").Anzahl.sum().equals(persistence_table.groupby("Verfahren").Nenner.first())
table(persistence_table,"06b_persistenz_rueckkehr","6b. Verlauf nach erster Nicht-Retained-Bewertung vor der letzten Stufe",
      "Nenner: eindeutig erstmals bei 500, 300 oder 150 Nicht-Retained bewertete Laufbahnen je Verfahren. "
      "Aggregation: Faktenzählung; erste Bewertung bei 75 und unklare erste Stufe ausgeschlossen. "
      "Fehlwerte: offen nur, wenn kein späteres Retained beobachtet wird und mindestens ein späterer Fehlwert vorliegt. "
      "Mindestens ein beobachtetes späteres Retained genügt für die Rückkehrkategorie, auch mit weiteren Fehlwerten.")


6a. Erste Nicht-Retained-Stufe und durchgehend Retained
    Verfahren                      Verlauf  Anzahl  Nenner Anteil_%
       Lead-k                 erstmals 500     487    2000  24.3500
       Lead-k                 erstmals 300     532    2000  26.6000
       Lead-k                 erstmals 150     388    2000  19.4000
       Lead-k                  erstmals 75     254    2000  12.7000
       Lead-k         durchgehend Retained     332    2000  16.6000
       Lead-k erstes Nicht-Retained unklar       7    2000   0.3500
        Plain                 erstmals 500     898    2000  44.9000
        Plain                 erstmals 300     122    2000   6.1000
        Plain                 erstmals 150     101    2000   5.0500
        Plain                  erstmals 75     229    2000  11.4500
        Plain         durchgehend Retained     634    2000  31.7000
        Plain erstes Nicht-Retained unklar      16    2000   0.8000
Zero-Shot-CoT                 erstmals 500     560    2000  

### 7. Entitätstypen und Daten hinter Abbildung 14

In [11]:
type_counts = facts.groupby("entity_type").agg(Fakten=("canonical_fact_id","nunique"),Artikel=("article_id","nunique")).reset_index().rename(columns={"entity_type":"Entitätstyp"})
assert type_counts.Fakten.sum()==2000
table(type_counts,"07a_entitaetstypen_umfang","7a. Ausgangsfakten und beteiligte Artikel je Entitätstyp",
      "Nenner: 2.000 einzigartige Ausgangsfakten aus 100 Artikeln. Aggregation: einzigartige Fakten und "
      "Artikel je Typ, ohne Vervielfachung durch Verfahren/Stufen; Artikel können mehrere Typen enthalten. "
      "Fehlwerte: kein Entitätstyp fehlt.")
validated_path = TABLES/f"05_verlustmuster/retention_entitaetstyp_methoden_stufen_95ci.csv"
record_source(validated_path, "Abgleich vorhandener Auswertung")
validated_entities = read(validated_path).set_index(["method","stage_target_words","entity_type"])
entity_rows = []
for m in METHODS:
    rows = []
    for t in STAGES:
        for typ in sorted(facts.entity_type.unique()):
            sub = qa[qa.method.eq(m)&qa.stage_target_words.eq(t)&qa.entity_type.eq(typ)]
            previous = validated_entities.loc[(m,t,typ)]
            s = {"count":int(previous.n_numerator),"den":int(previous.n_denominator),
                "mean":previous.estimate,"low":previous.ci_low,"high":previous.ci_high,
                "n_articles":int(previous.n_articles_denominator),
                "n_boot_valid":B-int(previous.bootstrap_draws_zero_denominator)}
            row = {"Verfahren":NAMES[m],"Wörter":t,"Entitätstyp":typ,"Retained":s["count"],"QA_gültig":s["den"],
                   "QA_fehlend":int(sub.technical_missing.sum()),"R_%":100*s["mean"],"KI_lo_%":100*s["low"],
                   "KI_hi_%":100*s["high"],"Artikel_gültig":s["n_articles"],"Boot_gültig":s["n_boot_valid"]}
            rows.append(row)
            entity_rows.append({"method":m,"stage_target_words":t,"entity_type":typ,**s})
    table(pd.DataFrame(rows),f"07_retention_typ_{m}",f"7b. Retention je Entitätstyp: {NAMES[m]}",
          "Nenner: gültige QA-Positionen des Typs/Verfahrens/der Stufe (QA_gültig). "
          "Aggregation: Summe Retained / Summe gültig, faktengewichtet; kein Mittel der Artikelquoten. "
          "Fehlwerte: technische QA-Urteile ausgeschlossen; Artikel ohne gültige Typ-Fakten tragen 0/0 Zählbeiträge bei. "
          "Bootstrap: alle 100 Artikel als Cluster; Ziehungen ohne gültigen Nenner sind fehlend, Boot_gültig angegeben.")

FIG14_METHODS = ["zero_shot_cot","auto_cot","automate_cot"]
fig_rows = []
selected = paths[paths.method.isin(FIG14_METHODS)]
for before,after in zip(STAGES[:-1],STAGES[1:]):
    valid_pair = selected[before].ne(M)&selected[after].ne(M)
    for typ in sorted(facts.entity_type.unique()):
        sub = selected[valid_pair&selected.entity_type.eq(typ)].copy()
        sub["risk"] = sub[before].eq(R)
        sub["loss"] = sub.risk&sub[after].isin(NON)
        s = estimate_ratio(sub,"loss","risk")
        fig_rows.append({"Übergang":f"{before}→{after}","Entitätstyp":typ,"Nicht_Retained_danach":s["count"],
                         "Retained_vorher_gültig":s["den"],"Verlust_%":100*s["mean"],"KI_lo_%":100*s["low"],
                         "KI_hi_%":100*s["high"],"Boot_gültig":s["n_boot_valid"]})
fig14 = pd.DataFrame(fig_rows)
table(fig14,"07c_abbildung14_verlustanteile","7c. Stufenspezifische Verlustanteile hinter Abbildung 14",
      "Nenner: zuvor Retained bewertete Fakten-Verfahrens-Läufe mit gültiger QA an beiden Endpunkten. "
      "Aggregation: Zero-Shot-CoT, Auto-CoT und Automate-CoT gepoolt; Zähler und Nenner über diese drei "
      "Verfahren summiert, kein ungewichtetes Mittel ihrer Prozentwerte. Eine Linie je Entitätstyp über "
      "500→300, 300→150 und 150→75. Zähler: Wechsel zu Lost/reference_discordant. "
      "Fehlwerte: Paar mit fehlender QA an einem Endpunkt ausgeschlossen; 0-Risiko-Bootstrap-Ziehung fehlend. "
      "Alle Methoden/Fakten eines gezogenen Artikels bleiben gemeinsam im Cluster.")


7a. Ausgangsfakten und beteiligte Artikel je Entitätstyp
Entitätstyp  Fakten  Artikel
  DATE_TIME     216       90
      EVENT      80       53
   LOCATION     255       93
     NUMBER     526       96
     OBJECT     168       67
        ORG     154       77
      OTHER     181       74
     PERSON     420       97

Nenner: 2.000 einzigartige Ausgangsfakten aus 100 Artikeln. Aggregation: einzigartige Fakten und Artikel
je Typ, ohne Vervielfachung durch Verfahren/Stufen; Artikel können mehrere Typen enthalten. Fehlwerte:
kein Entitätstyp fehlt.



7b. Retention je Entitätstyp: Lead-k
Verfahren  Wörter Entitätstyp  Retained  QA_gültig  QA_fehlend     R_% KI_lo_% KI_hi_%  Artikel_gültig  Boot_gültig
   Lead-k     500   DATE_TIME       158        215           1 73.4884 66.3962 80.1077              90        10000
   Lead-k     500       EVENT        65         80           0 81.2500 72.0430 89.8734              53        10000
   Lead-k     500    LOCATION       192        255           0 75.2941 69.7761 80.7087              93        10000
   Lead-k     500      NUMBER       423        526           0 80.4183 76.0331 84.5150              96        10000
   Lead-k     500      OBJECT       128        168           0 76.1905 68.9266 83.0882              67        10000
   Lead-k     500         ORG       117        154           0 75.9740 68.3871 82.9932              77        10000
   Lead-k     500       OTHER       131        180           1 72.7778 65.9341 79.3551              74        10000
   Lead-k     500      PERSON      

7b. Retention je Entitätstyp: Zero-Shot-CoT
    Verfahren  Wörter Entitätstyp  Retained  QA_gültig  QA_fehlend     R_% KI_lo_% KI_hi_%  Artikel_gültig  Boot_gültig
Zero-Shot-CoT     500   DATE_TIME       144        215           1 66.9767 60.2510 73.4884              90        10000
Zero-Shot-CoT     500       EVENT        60         80           0 75.0000 64.9111 84.1463              53        10000
Zero-Shot-CoT     500    LOCATION       166        254           1 65.3543 57.9707 72.5808              93        10000
Zero-Shot-CoT     500      NUMBER       413        526           0 78.5171 74.5490 82.4904              96        10000
Zero-Shot-CoT     500      OBJECT       122        167           1 73.0539 64.6256 81.2500              67        10000
Zero-Shot-CoT     500         ORG       118        154           0 76.6234 69.5035 83.2258              77        10000
Zero-Shot-CoT     500       OTHER       135        181           0 74.5856 67.0455 81.5920              74        10

7b. Retention je Entitätstyp: Few-Shot-CoT
   Verfahren  Wörter Entitätstyp  Retained  QA_gültig  QA_fehlend     R_% KI_lo_% KI_hi_%  Artikel_gültig  Boot_gültig
Few-Shot-CoT     500   DATE_TIME       121        214           2 56.5421 50.0000 63.1847              88        10000
Few-Shot-CoT     500       EVENT        47         79           1 59.4937 48.9118 70.1299              52        10000
Few-Shot-CoT     500    LOCATION       145        252           3 57.5397 50.0000 64.7303              92        10000
Few-Shot-CoT     500      NUMBER       383        526           0 72.8137 68.3190 77.1135              96        10000
Few-Shot-CoT     500      OBJECT       106        167           1 63.4731 53.8911 71.9179              67        10000
Few-Shot-CoT     500         ORG       113        154           0 73.3766 66.8790 79.7386              77        10000
Few-Shot-CoT     500       OTHER       120        178           3 67.4157 58.1001 76.1365              74        10000
Few-S

7b. Retention je Entitätstyp: Auto-CoT
Verfahren  Wörter Entitätstyp  Retained  QA_gültig  QA_fehlend     R_% KI_lo_% KI_hi_%  Artikel_gültig  Boot_gültig
 Auto-CoT     500   DATE_TIME       145        216           0 67.1296 60.4651 73.6111              90        10000
 Auto-CoT     500       EVENT        59         79           1 74.6835 64.8649 83.9506              52        10000
 Auto-CoT     500    LOCATION       165        254           1 64.9606 58.5938 71.0207              93        10000
 Auto-CoT     500      NUMBER       409        524           2 78.0534 74.1245 81.7845              96        10000
 Auto-CoT     500      OBJECT       121        166           2 72.8916 63.5800 82.0898              67        10000
 Auto-CoT     500         ORG       120        154           0 77.9221 70.7690 84.5247              77        10000
 Auto-CoT     500       OTHER       130        179           2 72.6257 65.3630 79.7982              73        10000
 Auto-CoT     500      PERSON    

7c. Stufenspezifische Verlustanteile hinter Abbildung 14
Übergang Entitätstyp  Nicht_Retained_danach  Retained_vorher_gültig Verlust_% KI_lo_% KI_hi_%  Boot_gültig
 500→300   DATE_TIME                     28                     430    6.5116  4.1001  9.1880        10000
 500→300       EVENT                     12                     169    7.1006  2.8844 12.1019        10000
 500→300    LOCATION                     41                     503    8.1511  5.0916 11.5460        10000
 500→300      NUMBER                     51                    1241    4.1096  2.7178  5.6620        10000
 500→300      OBJECT                     36                     375    9.6000  6.4327 13.4716        10000
 500→300         ORG                     22                     362    6.0773  3.4851  9.1812        10000
 500→300       OTHER                     36                     392    9.1837  6.2146 12.5874        10000
 500→300      PERSON                     79                     832    9.4952  7.1245 1

### 8. Weitere Evaluationsmetriken

In [12]:
METRICS = {"highlight_rouge1_f1":"ROUGE-1 F1", "highlight_rouge2_f1":"ROUGE-2 F1",
           "highlight_rougeL_f1":"ROUGE-L F1", "highlight_bertscore_f1":"BERTScore F1",
           "source_summac":"SummaC Original", "stage_input_summac":"SummaC Stufeneingang"}
metric_path = frozen_input(MET_DATA,VERSION+"/metrics_detailed.csv")
metric_manifest = frozen_input(MET_DATA,VERSION+"/metrics_manifest.json")
metric_source_manifest = json.loads((MET_DATA/"source_manifest.json").read_text())
assert hashlib.sha256(gzip.decompress((QA_DATA/VERSION/"summaries.csv.gz").read_bytes())).hexdigest() == metric_source_manifest["summary_original_sha256"]
metrics = read(metric_path)
assert len(metrics)==2400 and not metrics.duplicated(CELL).any()
assert set(metrics[CELL].itertuples(index=False,name=None)) == set(articles[CELL].itertuples(index=False,name=None))
for col in METRICS:
    metrics[col] = numeric(metrics[col])
metrics = metrics.merge(articles[CELL+["W","R_obs","empty_input","empty_output"]],on=CELL,
                        validate="one_to_one",suffixes=("","_qa"))
assert metrics.stage_actual_words.eq(metrics.W).all()
metrics["stage_input_summac_stored"] = metrics.stage_input_summac
metrics.loc[metrics.empty_input,"stage_input_summac"] = np.nan
assert metrics.empty_input.sum()==1
assert metrics.loc[metrics.stage_target_words.eq(500),"source_summac"].eq(metrics.loc[metrics.stage_target_words.eq(500),"stage_input_summac"]).all()
metrics.to_csv(OUT/"artikelwerte_sechs_metriken.csv",index=False,na_rep="fehlend")
validated_path = TABLES/f"06_weitere_metriken/mittelwerte_24_gruppen_alle_metriken_95ci.csv"
record_source(validated_path, "Abgleich vorhandener Auswertung")
validated_metrics = read(validated_path).set_index(["method","stage_target_words","metric"])
metric_rows = []
for metric,title in METRICS.items():
    rows = []
    for t in STAGES:
        for m in METHODS:
            sub = metrics[metrics.method.eq(m)&metrics.stage_target_words.eq(t)].set_index("article_id").reindex(IDS)
            previous = validated_metrics.loc[(m,t,metric)]
            s = {"mean":previous["mean"],"low":previous.ci_low,"high":previous.ci_high,
                "n":int(previous.n_valid),"n_boot_valid":B}
            rows.append({"Verfahren":NAMES[m],"Wörter":t,"MW":s["mean"],"KI_lo":s["low"],"KI_hi":s["high"],
                         "n_gültig":s["n"],"n_fehlend":N-s["n"]})
            metric_rows.append({"method":m,"stage_target_words":t,"metric":metric,**s})
    table(pd.DataFrame(rows),f"08_{metric}",f"8. {title}: Mittelwert und 95-%-KI",
          "Nenner: gültige Artikelwerte je Verfahren/Stufe (n_gültig), höchstens 100. Aggregation: "
          "gleichgewichtetes Artikelmittel auf Score-Skala; ROUGE/BERTScore gegen CNN/DailyMail-Highlights; "
          "SummaC gegen Original bzw. tatsächlichen unmittelbaren Stufeneingang. "
          "Fehlwerte: fehlende Scores ausgeschlossen; lokaler SummaC bei leerem Stufeneingang fehlend. "
          "Zwei leere Ausgaben behalten die dokumentierten End-to-end-Nullwerte. "
          "BERTScore/SummaC sind eingefrorene Modellwerte, kein neuer Modelllauf.",digits=6)


8. ROUGE-1 F1: Mittelwert und 95-%-KI
    Verfahren  Wörter       MW    KI_lo    KI_hi  n_gültig  n_fehlend
       Lead-k     500 0.166410 0.153932 0.179559       100          0
        Plain     500 0.271921 0.254564 0.289952       100          0
Zero-Shot-CoT     500 0.206166 0.191431 0.221358       100          0
 Few-Shot-CoT     500 0.221016 0.204604 0.237863       100          0
     Auto-CoT     500 0.204561 0.190162 0.219363       100          0
 Automate-CoT     500 0.189729 0.175982 0.204010       100          0
       Lead-k     300 0.235844 0.219202 0.252879       100          0
        Plain     300 0.281775 0.265150 0.299103       100          0
Zero-Shot-CoT     300 0.227974 0.212232 0.243777       100          0
 Few-Shot-CoT     300 0.234132 0.217731 0.250791       100          0
     Auto-CoT     300 0.222356 0.206840 0.238319       100          0
 Automate-CoT     300 0.217687 0.202116 0.233459       100          0
       Lead-k     150 0.337334 0.315984 0.358448    

8. BERTScore F1: Mittelwert und 95-%-KI
    Verfahren  Wörter       MW    KI_lo    KI_hi  n_gültig  n_fehlend
       Lead-k     500 0.880693 0.878319 0.883046       100          0
        Plain     500 0.890458 0.887943 0.893040       100          0
Zero-Shot-CoT     500 0.877061 0.875049 0.879134       100          0
 Few-Shot-CoT     500 0.877112 0.875209 0.879148       100          0
     Auto-CoT     500 0.877532 0.875671 0.879440       100          0
 Automate-CoT     500 0.877916 0.876114 0.879784       100          0
       Lead-k     300 0.881552 0.878975 0.884126       100          0
        Plain     300 0.896829 0.894281 0.899443       100          0
Zero-Shot-CoT     300 0.877473 0.875315 0.879728       100          0
 Few-Shot-CoT     300 0.877820 0.875769 0.880009       100          0
     Auto-CoT     300 0.878365 0.876492 0.880297       100          0
 Automate-CoT     300 0.878057 0.876214 0.879961       100          0
       Lead-k     150 0.913172 0.910677 0.915634  

### 9. Spearman-Zusammenhang mit QA-Retention

Separat innerhalb jedes Verfahrens und jeder Stufe über die Artikel, keine Korrelation der sechs Gruppenmittel.

In [13]:
def spearman_bootstrap(x,y):
    x,y = np.asarray(x,dtype=float),np.asarray(y,dtype=float)
    valid = np.isfinite(x)&np.isfinite(y)
    x,y = np.where(valid,x,np.nan),np.where(valid,y,np.nan)
    if valid.sum()<2 or np.unique(x[valid]).size<2 or np.unique(y[valid]).size<2:
        return {"rho":np.nan,"low":np.nan,"high":np.nan,"n":int(valid.sum()),"n_boot_valid":0},np.full(B,np.nan)
    rho = float(spearmanr(x[valid],y[valid]).statistic)
    boot = np.full(B,np.nan)
    # Re-rank each resampled article pair, including repeated articles and ties.
    for start in range(0,B,500):
        sample = draws[start:start+500]
        xr,yr = rankdata(x[sample],axis=1,method="average",nan_policy="omit"),rankdata(y[sample],axis=1,method="average",nan_policy="omit")
        n_valid = np.isfinite(xr).sum(axis=1)
        center_x = np.divide(np.nansum(xr,axis=1),n_valid,out=np.full(len(sample),np.nan),where=n_valid>0)
        center_y = np.divide(np.nansum(yr,axis=1),n_valid,out=np.full(len(sample),np.nan),where=n_valid>0)
        xr,yr = xr-center_x[:,None],yr-center_y[:,None]
        cross = np.nansum(xr*yr,axis=1)
        denominator = np.sqrt(np.nansum(xr*xr,axis=1)*np.nansum(yr*yr,axis=1))
        boot[start:start+len(sample)] = np.divide(cross,denominator,out=np.full(len(sample),np.nan),where=(denominator>0)&(n_valid>=2))
    lo,hi = percentile_interval(boot)
    return {"rho":rho,"low":lo,"high":hi,"n":int(valid.sum()),"n_boot_valid":int(np.isfinite(boot).sum())},boot

# Check resampled ranks against scipy on tied values and missing paired values.
test_x = np.arange(N,dtype=float)%7
test_y = np.arange(N,dtype=float)%11
test_x[3]=np.nan
_, test_boot = spearman_bootstrap(test_x,test_y)
for i in [0,1,32,100,9999]:
    sx,sy = test_x[draws[i]],test_y[draws[i]]
    valid = np.isfinite(sx)&np.isfinite(sy)
    assert np.isclose(test_boot[i],spearmanr(sx[valid],sy[valid]).statistic,rtol=0,atol=1e-12)
assert np.isnan(spearman_bootstrap(np.ones(N),np.arange(N))[0]["rho"])
correlation_rows, correlation_boot = [], {}
for metric,title in METRICS.items():
    rows = []
    for t in STAGES:
        for m in METHODS:
            sub = metrics[metrics.method.eq(m)&metrics.stage_target_words.eq(t)].set_index("article_id").reindex(IDS)
            s,correlation_boot[m,t,metric] = spearman_bootstrap(sub.R_obs,sub[metric])
            rows.append({"Verfahren":NAMES[m],"Wörter":t,"Spearman_ρ":s["rho"],"KI_lo":s["low"],"KI_hi":s["high"],
                         "n_Wertepaare":s["n"],"Boot_gültig":s["n_boot_valid"]})
            correlation_rows.append({"method":m,"stage_target_words":t,"metric":metric,**s})
    table(pd.DataFrame(rows),f"09_spearman_{metric}",f"9. QA-Retention (K/V) vs. {title}: Spearman",
          "Nenner: Artikel mit gültiger QA-Retention K/V und gültigem Metrikwert innerhalb der jeweiligen "
          "Verfahren-Stufe-Gruppe (n_Wertepaare). Aggregation: Spearman über Artikel, Mittelränge bei Ties. "
          "Fehlwerte: paarweise ausgeschlossen; ρ bei konstantem Rand oder weniger als zwei Paaren fehlend. "
          "Bootstrap: gemeinsame Artikelpaare, Ränge in jeder Ziehung neu; undefinierte Ziehungen ausgeschlossen, "
          "Boot_gültig von 10.000 angegeben. Keine Korrelation über Verfahren oder Stufen gepoolt.",digits=6)


9. QA-Retention (K/V) vs. ROUGE-1 F1: Spearman
    Verfahren  Wörter Spearman_ρ     KI_lo    KI_hi  n_Wertepaare  Boot_gültig
       Lead-k     500   0.321615  0.138148 0.481607           100        10000
        Plain     500   0.177690 -0.017418 0.361031           100        10000
Zero-Shot-CoT     500   0.163811 -0.030695 0.352544           100        10000
 Few-Shot-CoT     500   0.191448 -0.004951 0.378596           100        10000
     Auto-CoT     500   0.171957 -0.022797 0.358833           100        10000
 Automate-CoT     500   0.093496 -0.106969 0.288946           100        10000
       Lead-k     300   0.284937  0.092183 0.461286           100        10000
        Plain     300   0.240773  0.045859 0.420616           100        10000
Zero-Shot-CoT     300   0.211597  0.010867 0.402952           100        10000
 Few-Shot-CoT     300   0.216138  0.018377 0.398600           100        10000
     Auto-CoT     300   0.185354 -0.016196 0.370365           100        10000
 Auto

9. QA-Retention (K/V) vs. ROUGE-2 F1: Spearman
    Verfahren  Wörter Spearman_ρ     KI_lo    KI_hi  n_Wertepaare  Boot_gültig
       Lead-k     500   0.384620  0.202568 0.544280           100        10000
        Plain     500   0.232819  0.042898 0.406051           100        10000
Zero-Shot-CoT     500   0.215793  0.017331 0.406066           100        10000
 Few-Shot-CoT     500   0.185871 -0.015032 0.375316           100        10000
     Auto-CoT     500   0.193570  0.004474 0.375454           100        10000
 Automate-CoT     500   0.201710  0.004948 0.382846           100        10000
       Lead-k     300   0.367568  0.176391 0.537097           100        10000
        Plain     300   0.246206  0.055944 0.417261           100        10000
Zero-Shot-CoT     300   0.264648  0.070756 0.444133           100        10000
 Few-Shot-CoT     300   0.234702  0.038342 0.414819           100        10000
     Auto-CoT     300   0.223225  0.027689 0.402439           100        10000
 Auto

9. QA-Retention (K/V) vs. ROUGE-L F1: Spearman
    Verfahren  Wörter Spearman_ρ     KI_lo    KI_hi  n_Wertepaare  Boot_gültig
       Lead-k     500   0.378095  0.199502 0.533881           100        10000
        Plain     500   0.047960 -0.142966 0.239283           100        10000
Zero-Shot-CoT     500   0.166771 -0.026099 0.354476           100        10000
 Few-Shot-CoT     500   0.062362 -0.138190 0.260280           100        10000
     Auto-CoT     500   0.136598 -0.057841 0.325952           100        10000
 Automate-CoT     500   0.083989 -0.108649 0.277645           100        10000
       Lead-k     300   0.353405  0.162732 0.527109           100        10000
        Plain     300   0.098856 -0.087780 0.281046           100        10000
Zero-Shot-CoT     300   0.192755 -0.009514 0.379559           100        10000
 Few-Shot-CoT     300   0.111459 -0.090929 0.304720           100        10000
     Auto-CoT     300   0.160764 -0.037690 0.347063           100        10000
 Auto

9. QA-Retention (K/V) vs. BERTScore F1: Spearman
    Verfahren  Wörter Spearman_ρ     KI_lo     KI_hi  n_Wertepaare  Boot_gültig
       Lead-k     500   0.328149  0.133003  0.503017           100        10000
        Plain     500  -0.277208 -0.444006 -0.092422           100        10000
Zero-Shot-CoT     500   0.057221 -0.140686  0.261892           100        10000
 Few-Shot-CoT     500  -0.024487 -0.212732  0.166987           100        10000
     Auto-CoT     500   0.097404 -0.105851  0.299890           100        10000
 Automate-CoT     500   0.074221 -0.119042  0.265155           100        10000
       Lead-k     300   0.262193  0.066239  0.446200           100        10000
        Plain     300  -0.234225 -0.399355 -0.059148           100        10000
Zero-Shot-CoT     300  -0.011180 -0.209042  0.195283           100        10000
 Few-Shot-CoT     300   0.010492 -0.186167  0.206947           100        10000
     Auto-CoT     300   0.043205 -0.168177  0.246996           100     

9. QA-Retention (K/V) vs. SummaC Original: Spearman
    Verfahren  Wörter Spearman_ρ     KI_lo    KI_hi  n_Wertepaare  Boot_gültig
       Lead-k     500   0.014235 -0.211723 0.237410           100        10000
        Plain     500  -0.029584 -0.229419 0.170049           100        10000
Zero-Shot-CoT     500   0.061371 -0.144438 0.264256           100        10000
 Few-Shot-CoT     500   0.255002  0.058598 0.437855           100        10000
     Auto-CoT     500   0.023576 -0.186160 0.225004           100        10000
 Automate-CoT     500  -0.015557 -0.225702 0.188473           100        10000
       Lead-k     300   0.027790 -0.180131 0.228880           100        10000
        Plain     300  -0.025931 -0.234832 0.185989           100        10000
Zero-Shot-CoT     300   0.132472 -0.065758 0.324006           100        10000
 Few-Shot-CoT     300   0.191819 -0.008699 0.382929           100        10000
     Auto-CoT     300  -0.002117 -0.204245 0.196952           100        10000


9. QA-Retention (K/V) vs. SummaC Stufeneingang: Spearman
    Verfahren  Wörter Spearman_ρ     KI_lo     KI_hi  n_Wertepaare  Boot_gültig
       Lead-k     500   0.014235 -0.211723  0.237410           100        10000
        Plain     500  -0.029584 -0.229419  0.170049           100        10000
Zero-Shot-CoT     500   0.061371 -0.144438  0.264256           100        10000
 Few-Shot-CoT     500   0.255002  0.058598  0.437855           100        10000
     Auto-CoT     500   0.023576 -0.186160  0.225004           100        10000
 Automate-CoT     500  -0.015557 -0.225702  0.188473           100        10000
       Lead-k     300   0.038071 -0.178223  0.249464           100        10000
        Plain     300   0.126571 -0.068510  0.307667           100        10000
Zero-Shot-CoT     300  -0.022590 -0.230781  0.183950           100        10000
 Few-Shot-CoT     300  -0.080706 -0.269741  0.110574           100        10000
     Auto-CoT     300   0.149804 -0.049466  0.332818           

### 10. QA-Kalibrierung gegenüber menschlichen Referenzlabels

In [14]:
calibration_items = None
if (VAL_DATA/"qa/internal_model_label_key.csv").is_file():
    human_key_path = frozen_input(VAL_DATA,"qa/internal_model_label_key.csv")
    human_key = read(human_key_path)
    assert len(human_key)==60 and human_key.review_id.is_unique
    human_returns = []
    for path in sorted((VAL_DATA/"qa/returns").glob("reviewer_*_answers.csv")):
        path = frozen_input(VAL_DATA,str(path.relative_to(VAL_DATA)))
        frame = read(path)
        assert len(frame)==60 and frame.review_id.is_unique and set(frame.review_id)==set(human_key.review_id)
        frame["human_answer_label"] = frame.human_answer_label.str.strip().str.lower().str.replace(" ","_",regex=False).str.replace("-","_",regex=False)
        assert frame.human_answer_label.isin(["retained","lost","unsupported","unclear",""]).all()
        human_returns.append(frame[["review_id","human_answer_label"]])
    votes = pd.concat(human_returns,ignore_index=True)
    consensus_rows = []
    for review_id,group in votes.groupby("review_id"):
        count = Counter(group.loc[group.human_answer_label.ne(""),"human_answer_label"])
        winners = sorted(label for label,n in count.items() if n==max(count.values())) if count else []
        consensus_rows.append({"review_id":review_id,"human_label":winners[0] if winners else "",
                               "tie":len(winners)>1,"n_votes":sum(count.values())})
    calibration_items = human_key.merge(pd.DataFrame(consensus_rows),on="review_id",validate="one_to_one")
    def original_retained(reference,answer):
        if answer.upper()=="NOT_FOUND": return False
        r,a = scorer.normalize(reference),scorer.normalize(answer)
        return bool(r and (r in a or a in r)) or scorer.token_f1(reference,answer)>=.50
    def classify_human(reference,answer,fn):
        return "lost" if answer.upper()=="NOT_FOUND" else "retained" if fn(reference,answer) else "unsupported"
    for col,fn in [("original",original_retained),("final",scorer.retained)]:
        calibration_items[col] = [classify_human(r,a,fn) for r,a in calibration_items[["reference_answer","model_answer"]].itertuples(index=False,name=None)]
    assert calibration_items.original.eq(calibration_items.model_label).all()
    valid = calibration_items.human_label.isin(["retained","lost","unsupported"])&~calibration_items.tie
    strict = calibration_items[valid].copy()
    assert len(strict)==52
    labels = ["retained","lost","unsupported"]
    recall_rows,calibration_summary = [],[]
    for column,title in [("original","Ursprünglicher Scorer"),("final","Finaler Scorer")]:
        matrix = pd.crosstab(strict.human_label,strict[column]).reindex(index=labels,columns=labels,fill_value=0)
        matrix.index.name="Menschliche_Referenz"
        table(matrix.reset_index(),f"10_konfusion_{column}",f"10. Konfusionsmatrix: {title}",
              "Nenner: dieselben 52 eindeutigen Kalibrierungsfälle, Zeilen menschliche Konsenslabels, "
              "Spalten automatische Labels. Aggregation: ein Fall = eine Zählung; menschlicher Konsens "
              "aus verfügbaren Reviewer-Stimmen. Fehlwerte: fehlende Stimmen nicht ergänzt; sechs "
              "Antwort-Ties und zwei unclear-Konsensfälle ausgeschlossen. Keine unabhängigen Prüffälle.")
        for label in labels:
            correct,total = int(matrix.loc[label,label]),int(matrix.loc[label].sum())
            recall_rows.append({"Scorer":title,"Referenzklasse":label,"Korrekt":correct,"Gesamt":total,
                                "Recall_%":100*correct/total if total else np.nan})
        calibration_summary.append({"Scorer":title,"Korrekt":int(np.trace(matrix)),"Fälle":len(strict),
                                    "Accuracy_%":100*np.trace(matrix)/len(strict),"Set":"Kalibrierungsfälle"})
    recall_table = pd.DataFrame(recall_rows)
    calibration_summary = pd.DataFrame(calibration_summary)
    table(recall_table,"10_recall","10. Je menschlicher Referenzklasse: korrekt / gesamt und Recall",
          "Nenner: Anzahl der Fälle in der jeweiligen menschlichen Referenzklasse, nicht Anzahl "
          "automatischer Vorhersagen. Aggregation: Diagonale der Konfusionsmatrix / Zeilensumme. "
          "Fehlwerte: gleiche Auswahl der 52 eindeutigen Fälle; Recall bei leerer Referenzklasse fehlend. "
          "Historisches unsupported ist eine Human-Referenzklasse, kein separates Quelllabel des Hauptexperiments.")
    table(calibration_summary,"10_kalibrierungsbasis","10. Auswertungsbasis der Scorerleistung",
          "Nenner: dieselben 52 Kalibrierungsfälle aus sechs Artikeln (von 60 ausgewählten Fällen). "
          "Aggregation: korrekte Fallklassifikationen / Fälle. Fehlwerte: Ties und unclear ausgeschlossen. "
          "Die Scorerregeln wurden an diesen Fällen entwickelt; diese Accuracy ist keine unabhängige "
          "Testleistung und validiert nicht die gesamte finale Faktgenerator-/QA-Pipeline.")
    calibration_items.to_csv(OUT/"kalibrierung_fallebene.csv",index=False)
else:
    table(pd.DataFrame([{"Ergebnis":"fehlend","Grund":"Human-Validierungsdaten nicht mitgeliefert"}]),
          "10_kalibrierung_fehlend","10. QA-Kalibrierung",
          "Nenner: nicht verfügbar. Aggregation: nicht berechenbar. Fehlwerte: keine Validierungsdaten vorhanden.")


10. Konfusionsmatrix: Ursprünglicher Scorer
Menschliche_Referenz  retained  lost  unsupported
            retained        16     0           14
                lost         1    16            2
         unsupported         0     3            0

Nenner: dieselben 52 eindeutigen Kalibrierungsfälle, Zeilen menschliche Konsenslabels, Spalten
automatische Labels. Aggregation: ein Fall = eine Zählung; menschlicher Konsens aus verfügbaren Reviewer-
Stimmen. Fehlwerte: fehlende Stimmen nicht ergänzt; sechs Antwort-Ties und zwei unclear-Konsensfälle
ausgeschlossen. Keine unabhängigen Prüffälle.

10. Konfusionsmatrix: Finaler Scorer
Menschliche_Referenz  retained  lost  unsupported
            retained        25     0            5
                lost         1    16            2
         unsupported         0     3            0

Nenner: dieselben 52 eindeutigen Kalibrierungsfälle, Zeilen menschliche Konsenslabels, Spalten
automatische Labels. Aggregation: ein Fall = eine Zählung; menschlicher K

## Prüfungen und Quellen

Die neuen Berechnungen werden mit den vorhandenen, ungerundeten Ergebnissen abgeglichen. Die gespeicherten Tabellen werden hierfür nicht als Recheneingaben verwendet.

In [15]:
def checked_export(relative):
    path = TABLES/relative.removeprefix("tables/")
    record_source(path,"Abgleich vorhandener Auswertung")
    return read(path)

archived_labels = checked_export(f"tables/03_qa_retention/qa_labels_reproduziert.csv.gz")
assert not archived_labels.duplicated(KEY).any() and len(archived_labels)==len(qa)
aligned = qa[KEY+["label"]].merge(archived_labels[KEY+["label"]],on=KEY,validate="one_to_one",suffixes=("","_archiv"))
assert aligned.label.eq(aligned.label_archiv).all()
validation_checks.append({"Prüfung":"QA-Labels positionsgenau reproduziert","Anzahl":len(aligned),"Status":"bestanden"})

archived_retention = checked_export(f"tables/04_laenge_und_adjustierte_retention/retention_und_drei_rangfolgen_24_gruppen.csv")
for row in archived_retention.itertuples():
    for metric,old in [("R_obs","R_obs"),("R_20","R_zero"),("R_adj","R_adj")]:
        s = retention_stats[row.method,row.stage_target_words,metric]
        for field,suffix in [("mean","mean"),("low","ci_low"),("high","ci_high")]:
            assert np.isclose(s[field],getattr(row,f"{old}_{suffix}"),rtol=0,atol=1e-12)
        assert rank(row.method,row.stage_target_words,metric)==getattr(row,f"rank_{old}")
validation_checks.append({"Prüfung":"Retentionsmittel, Intervalle und Ränge","Anzahl":24*3,"Status":"bestanden"})

entity_archive = checked_export(f"tables/05_verlustmuster/retention_entitaetstyp_methoden_stufen_95ci.csv")
entities_new = pd.DataFrame(entity_rows).merge(entity_archive,on=["method","stage_target_words","entity_type"],validate="one_to_one")
for new,old in [("mean","estimate"),("low","ci_low"),("high","ci_high")]:
    assert np.allclose(entities_new[new],entities_new[old],rtol=0,atol=1e-12,equal_nan=True)
validation_checks.append({"Prüfung":"Entitätsretention und Intervalle","Anzahl":len(entities_new),"Status":"bestanden"})

fig_archive = checked_export(f"tables/05_verlustmuster/verlust_entitaetstyp_uebergaenge_95ci.csv")
fig_archive = fig_archive[fig_archive.method_group.eq("Drei generative Verfahren gepoolt")]
fig_aligned = fig14.merge(fig_archive,left_on=["Übergang","Entitätstyp"],right_on=["transition","entity_type"],validate="one_to_one")
for new,old in [("Verlust_%","loss_percent"),("KI_lo_%","loss_ci_low_percent"),("KI_hi_%","loss_ci_high_percent")]:
    assert np.allclose(fig_aligned[new],fig_aligned[old],rtol=0,atol=1e-10,equal_nan=True)
validation_checks.append({"Prüfung":"Abbildung-14-Punktwerte und Intervalle","Anzahl":len(fig_aligned),"Status":"bestanden"})

metric_archive = checked_export(f"tables/06_weitere_metriken/mittelwerte_24_gruppen_alle_metriken_95ci.csv")
metric_aligned = pd.DataFrame(metric_rows).merge(metric_archive,on=["method","stage_target_words","metric"],validate="one_to_one",suffixes=("","_archiv"))
for new,old in [("mean","mean_archiv"),("low","ci_low"),("high","ci_high"),("n","n_valid")]:
    assert np.allclose(metric_aligned[new],numeric(metric_aligned[old]),rtol=0,atol=1e-12,equal_nan=True)
validation_checks.append({"Prüfung":"Sechs Metriken: Mittel, Intervalle und gültiges n","Anzahl":len(metric_aligned),"Status":"bestanden"})

path_archive = checked_export(f"tables/05_verlustmuster/faktenlaufbahnen.csv")
path_aligned = paths.merge(path_archive[PATHKEY+["first_category","persistence"]],on=PATHKEY,validate="one_to_one")
assert path_aligned.first.eq(path_aligned.first_category.replace({"always_retained":"durchgehend","undetermined":"offen"})).all()
assert path_aligned.loc[path_aligned.first.isin(["500","300","150"]),"later"].eq(
    path_aligned.loc[path_aligned.first.isin(["500","300","150"]),"persistence"].replace(
        {"all_later_non_retained":"durchgehend_nicht_Retained","observed_return":"später_Retained","persistence_undetermined":"offen"})).all()
validation_checks.append({"Prüfung":"Erste Verluststufe und Folgeverläufe","Anzahl":len(path_aligned),"Status":"bestanden"})
if calibration_items is not None:
    assert calibration_summary.Korrekt.tolist()==[32,41]
    validation_checks.append({"Prüfung":"Ursprünglicher/finaler Scorer: 32/52 und 41/52","Anzahl":52,"Status":"bestanden"})
validation_checks.append({"Prüfung":"Spearman-Resampling mit Ties/Fehlwerten vs. scipy","Anzahl":5,"Status":"bestanden"})
table(pd.DataFrame(validation_checks),"11_rechenpruefungen","Rechenprüfungen",
      "Nenner: jeweiliger Prüfumfang. Aggregation: vollständiger Schlüsselabgleich bzw. "
      "numerischer Vergleich mit absoluter Toleranz 10⁻¹² (Prozentwerte 10⁻¹⁰). "
      "Fehlwerte: bei numerischen Vergleichen nur identische fehlende Werte akzeptiert.")
source_index = pd.DataFrame(sources)[["Datei","Rolle"]]
table(source_index,"12_quellenindex","Quellenindex",
      "Nenner: tatsächlich gelesene Quellen einschließlich Manifeste und Abgleiche. Aggregation: "
      "eine Zeile pro Datei; vollständige SHA-256-Werte in quellen_sha256.csv. "
      "Fehlwerte: erforderliche Hauptquellen dürfen nicht fehlen; Validierungsquellen optional.")
source_frame = pd.DataFrame(sources)
source_frame.to_csv(OUT/"quellen_sha256.csv",index=False)
software = {package:importlib.metadata.version(package) for package in ["numpy","pandas","scipy","nbformat","nbclient","nbconvert","ipykernel"]}
manifest = {"version":VERSION,"n_articles":N,"n_outputs":len(articles),"n_qa":len(qa),
            "seed":SEED,"bootstrap_draws":B,"bootstrap_unit":"article", "interval":"95% percentile, linear quantiles",
            "paired":"one shared article index matrix across all methods, stages and facts",
            "spearman":"complete pairs within method-stage; average ranks recomputed in every draw",
            "retention_denominators":{"R_obs":"K/V","R_20":"K/20","R_adj":"(K/20)/(1+D)"},
            "length_penalty":"mean(R_20-R_adj)*100 percentage points",
            "ranking":"exact rational means; competition ranks for ties",
            "figure14_methods":FIG14_METHODS,"figure14_denominator":"retained at start and valid at both endpoints",
            "calibration":"same development cases, unique plurality excluding ties/unclear; no independent test",
            "software":{"python":platform.python_version(),**software},"tables":table_registry,
            "checks":validation_checks,"sources":sources}
(OUT/"execution_manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
pass  # Große Bootstrap-Zwischendatei wird nicht benötigt.
print(f"Alle Rechenprüfungen bestanden. {len(table_registry)} vollständige Texttabellen.\n"
      f"Text: {TEXT_FILE}\nCSV und Provenienz: {OUT}")


Rechenprüfungen
                                          Prüfung  Anzahl    Status
            QA-Labels positionsgenau reproduziert   48000 bestanden
           Retentionsmittel, Intervalle und Ränge      72 bestanden
                 Entitätsretention und Intervalle     192 bestanden
           Abbildung-14-Punktwerte und Intervalle      24 bestanden
Sechs Metriken: Mittel, Intervalle und gültiges n     144 bestanden
             Erste Verluststufe und Folgeverläufe   12000 bestanden
   Ursprünglicher/finaler Scorer: 32/52 und 41/52      52 bestanden
Spearman-Resampling mit Ties/Fehlwerten vs. scipy       5 bestanden

Nenner: jeweiliger Prüfumfang. Aggregation: vollständiger Schlüsselabgleich bzw. numerischer Vergleich
mit absoluter Toleranz 10⁻¹² (Prozentwerte 10⁻¹⁰). Fehlwerte: bei numerischen Vergleichen nur identische
fehlende Werte akzeptiert.

Quellenindex
                                                                                  Datei                           Rolle
  

## Befunde aus der ausgeführten Rechnung

In [16]:
from IPython.display import display, Markdown
missing = int(qa.technical_missing.sum())
lines = [f"- **QA:** {missing:,} technische Fehlwerte von {len(qa):,} Positionen ({100*missing/len(qa):.4f} %).",
         f"- **Nennerwechsel:** {int(changes.Positionen_Nennerwechsel.sum())} von 24 Methodenrangpositionen, "
         f"{int(changes.Positionen_Nennerwechsel.gt(0).sum())} von vier Stufen betroffen.",
         f"- **Längenadjustierung:** {int(changes.Positionen_Längenadjustierung.sum())} von 24 Methodenrangpositionen, "
         f"{int(changes.Positionen_Längenadjustierung.gt(0).sum())} von vier Stufen betroffen.",
         f"- **Lokaler SummaC:** Ein fehlender Wert bei leerem Stufeneingang; die betroffene Gruppe und Korrelation haben 99 gültige Artikelpaare."]
if calibration_items is not None:
    before,after = calibration_summary.iloc[0],calibration_summary.iloc[1]
    lines.append(f"- **Kalibrierung:** ursprünglich {int(before.Korrekt)}/52 ({before['Accuracy_%']:.4f} %), "
                 f"final {int(after.Korrekt)}/52 ({after['Accuracy_%']:.4f} %); Leistung auf denselben Kalibrierungsfällen.")
lines.append("- Kopierbare Texttabellen sind im Notebook gespeichert; CSV-Dateien enthalten ungerundete Werte. "
             "Die ausgewählten Spitzenvergleiche sind deskriptiv und keine unabhängig bestätigten Sieger.")
display(Markdown("\n".join(lines)))
with TEXT_FILE.open("a",encoding="utf-8") as stream:
    stream.write("Befunde\n"+"\n".join(lines)+"\n")


- **QA:** 196 technische Fehlwerte von 48,000 Positionen (0.4083 %).
- **Nennerwechsel:** 0 von 24 Methodenrangpositionen, 0 von vier Stufen betroffen.
- **Längenadjustierung:** 11 von 24 Methodenrangpositionen, 3 von vier Stufen betroffen.
- **Lokaler SummaC:** Ein fehlender Wert bei leerem Stufeneingang; die betroffene Gruppe und Korrelation haben 99 gültige Artikelpaare.
- **Kalibrierung:** ursprünglich 32/52 (61.5385 %), final 41/52 (78.8462 %); Leistung auf denselben Kalibrierungsfällen.
- Kopierbare Texttabellen sind im Notebook gespeichert; CSV-Dateien enthalten ungerundete Werte. Die ausgewählten Spitzenvergleiche sind deskriptiv und keine unabhängig bestätigten Sieger.